In [2]:
import pandas as pd
from geographiclib.geodesic import Geodesic
import geopandas
import math
import folium
from shapely.geometry import Point
from shapely.geometry import Polygon
import numpy as np
import matplotlib.pyplot as plt

In [70]:
geod = Geodesic.WGS84 

data = 'obc_tanker_cargo.csv'
df = pd.read_csv(data)
df = df[['MMSI', 'NAME', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'NAV_STATUS', 'SHIP_AND_CARGO_TYPE', 'DRAUGHT', 'DIM_BOW', 'DIM_STERN', 'DIM_PORT', 'DIM_STARBOARD']]
df['BEAM'] = df['DIM_PORT'] + df['DIM_STARBOARD']
df['LENGTH'] = df['DIM_BOW'] + df['DIM_STERN']
df = df[df['BEAM'] > 0]
df = df[df['LENGTH'] > 0]
df = df[df['DRAUGHT'] > 0]
df = df[df['SPEED_KNOTS'] > 3]
df = df.drop_duplicates(subset=['PERIOD', 'MMSI'])
df

,MMSI,NAME,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,COG_DEG,HEADING_DEG,NAV_STATUS,SHIP_AND_CARGO_TYPE,DRAUGHT,DIM_BOW,DIM_STERN,DIM_PORT,DIM_STARBOARD,BEAM,LENGTH
0,323150000,LPG EMILIA,22.779250,-78.741867,2023-04-01 00:00:00.000,14.7,110.0,112.0,0.0,80,8.4,143,35,14,14,28,178
1,334976000,BELEN,23.218183,-78.842584,2023-04-01 00:00:00.000,10.2,323.4,322.0,0.0,70,3.6,80,10,5,9,14,90
2,319103800,STOLT SINCERITY,22.423079,-77.790762,2023-04-01 00:00:00.000,13.7,302.6,302.0,0.0,80,10.5,155,30,18,14,32,185
3,323150000,LPG EMILIA,22.701027,-78.515972,2023-04-01 00:55:00.000,14.8,109.0,110.0,0.0,80,8.4,143,35,14,14,28,178
4,334979000,SARA REGINA,21.862225,-76.873167,2023-04-01 00:55:00.000,10.9,294.0,NaN,0.0,70,3.8,79,12,7,7,14,91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131303,677087000,HIMALAYA,21.740143,-76.850582,2025-09-12 23:10:00.000,5.1,295.2,NaN,0.0,70,3.0,78,5,5,7,12,83
131304,219484000,HAFNIA LIBRA,22.592591,-78.073797,2025-09-12 23:20:00.000,13.8,303.4,303.0,0.0,89,7.5,150,33,20,12,32,183
131305,255915671,BOCHEM ROTTERDAM,22.885718,-79.046140,2025-09-12 23:25:00.000,13.8,111.1,112.0,0.0,80,8.2,133,26,8,19,27,159
131306,338302000,MAGNOLIA STATE,21.888333,-77.188660,2025-09-12 23:45:00.000,15.2,132.3,132.0,0.0,80,9.5,151,35,19,13,32,186


In [4]:
df['SPEED_KNOTS'].HI

3541

In [71]:
df['PERIOD'] = pd.to_datetime(df['PERIOD'])
df['channel_side'] = ['Northwestbound' if x > 200 else 'Southeastbound' for x in df['COG_DEG']]
df = df.sort_values(['MMSI', 'PERIOD']).reset_index(drop=True)
df['time_diff'] = (df.groupby('MMSI')['PERIOD'].diff().dt.total_seconds())
#df['cog_diff'] = (df.groupby('MMSI')['COG_DEG'].diff())
#df['cog_diff'] = np.abs((df['cog_diff'] + 180) % 360 -180)

threshold = 30 * 60
df['new_voyage'] = (df['time_diff'] > threshold) | (df['time_diff'] < 0 ) | (df['time_diff'].isna()) #| (df['cog_diff'] > 75)
df['voyage_id'] = (df.groupby('MMSI')['new_voyage'].cumsum())
df['voyage_id'] = df['voyage_id'].astype('str') + '_' + df['MMSI'].astype('str')
df

,MMSI,NAME,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,COG_DEG,HEADING_DEG,NAV_STATUS,SHIP_AND_CARGO_TYPE,...,DIM_BOW,DIM_STERN,DIM_PORT,DIM_STARBOARD,BEAM,LENGTH,channel_side,time_diff,new_voyage,voyage_id
0,205042000,DELOS,22.736758,-78.616947,2023-06-04 14:00:00,13.0,110.0,111.0,0.0,80,...,281,55,30,30,60,336,Southeastbound,NaN,True,1_205042000
1,205042000,DELOS,22.707576,-78.536280,2023-06-04 14:20:00,13.1,111.2,112.0,0.0,80,...,281,55,30,30,60,336,Southeastbound,1200.0,False,1_205042000
2,205042000,DELOS,22.705818,-78.531350,2023-06-04 14:25:00,13.1,111.3,112.0,0.0,80,...,281,55,30,30,60,336,Southeastbound,300.0,False,1_205042000
3,205042000,DELOS,22.666790,-78.419349,2023-06-04 14:55:00,13.3,110.4,111.0,0.0,80,...,281,55,30,30,60,336,Southeastbound,1800.0,False,1_205042000
4,205042000,DELOS,22.654898,-78.384704,2023-06-04 15:05:00,13.4,110.2,111.0,0.0,80,...,281,55,30,30,60,336,Southeastbound,600.0,False,1_205042000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126554,720202000,INTL VICTORY,23.360848,-78.953454,2024-10-28 21:30:00,7.0,342.5,NaN,0.0,79,...,10,23,7,5,12,33,Northwestbound,26100.0,True,172_720202000
126555,720202000,INTL VICTORY,23.364144,-78.954727,2024-10-28 21:35:00,6.9,346.9,NaN,0.0,79,...,10,23,7,5,12,33,Northwestbound,300.0,False,172_720202000
126556,720202000,INTL VICTORY,22.251636,-77.465667,2024-11-15 21:30:00,6.3,130.8,NaN,0.0,79,...,10,23,7,5,12,33,Southeastbound,1554900.0,True,173_720202000
126557,720202000,INTL VICTORY,22.009253,-77.023147,2024-11-16 02:10:00,6.3,116.5,NaN,0.0,79,...,10,23,7,5,12,33,Southeastbound,16800.0,True,174_720202000


In [412]:
df.to_csv('clean_navcen_obc_datav2.csv')

In [54]:
print(df['MMSI'].nunique())
print(df['voyage_id'].nunique())
print(df.shape)

3541
56208
(126559, 23)


In [355]:
df.shape

(126559, 25)

In [64]:
data = pd.read_csv('obc_boundaries.csv')
data['geometry'] = data.apply(lambda x: (float(x.Long), float(x.Lat)), axis=1)
channel_coords = list(data['geometry'])
channel_coords_north = [channel_coords[0], channel_coords[1], channel_coords[2], channel_coords[3], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
channel_coords_south = [channel_coords[4], channel_coords[5], channel_coords[6], channel_coords[7], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
coords_north = [channel_coords[0], channel_coords[1], channel_coords[2], (-77.44235, 22.18378333), (-77.44235, 23.80836667), (-78.72296667, 23.80836667)]
coords_south = [channel_coords[4], channel_coords[5], channel_coords[6], channel_coords[7], (-77.4681,21.15348333), (-78.75143333, 21.15348333)]
channel_polygon_n = Polygon(channel_coords_north)
channel_polygon_s = Polygon(channel_coords_south)
polygon_n = Polygon(coords_north)
polygon_s = Polygon(coords_south)

In [68]:
def where_is_vessel(LAT_AVG, LON_AVG, BOOL_northc, BOOL_southc, BOOL_north, BOOL_south):
    if BOOL_northc == True:
        answer = 'in north channel'
    elif BOOL_southc == True:
        answer = 'in south channel'
    elif LON_AVG < -78.72296667:
        answer = 'west'
    elif LON_AVG > -77.49385:
        answer = 'east'
    elif BOOL_north == True:
        answer = 'north'
    elif BOOL_south == True:
        answer = 'south'
    else:
        answer = 'other'
    return answer

In [72]:
geo_df = geopandas.GeoDataFrame(df, geometry=geopandas.points_from_xy(df['LON_AVG'], df['LAT_AVG']), crs='EPSG:4326')
geo_df['in_channel_north'] = geo_df.within(channel_polygon_n)
geo_df['in_channel_south'] = geo_df.within(channel_polygon_s)
geo_df['north_of_channel'] = geo_df.within(polygon_n)
geo_df['south_of_channel'] = geo_df.within(polygon_s)
geo_df['location'] = geo_df.apply(lambda x: where_is_vessel(x.LAT_AVG, x.LON_AVG, x.in_channel_north, x.in_channel_south, x.north_of_channel, x.south_of_channel), axis=1)
geo_df.shape

(126559, 27)

In [73]:
geo_df['location'].value_counts()

location
east                38990
in south channel    36181
west                24665
in north channel    22403
north                3787
south                 533
Name: count, dtype: int64

In [74]:
geo_df['num_pings'] = geo_df['voyage_id'].map(geo_df['voyage_id'].value_counts())
geo_df = geo_df[geo_df['num_pings'] >= 5]

In [79]:
geo_df.to_csv('mid_datav2.csv')

In [76]:
geo_df2 = geo_df[((geo_df['in_channel_north'] == True) & (geo_df['channel_side'] == 'Northwestbound')) |
    ((geo_df['in_channel_south'] == True) & (geo_df['channel_side'] == 'Southeastbound'))]

In [363]:
geo_df2

(58312, 34)

In [77]:
geo_df2['num_pings'] = geo_df2['voyage_id'].map(geo_df2['voyage_id'].value_counts())
geo_df2 = geo_df2[geo_df2['num_pings'] >= 5]

/home/orda2/anaconda3/envs/vandy_capstone/lib/python3.13/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [78]:
geo_df2.to_csv('golden_datasetv5.csv')

In [129]:
dis = df[(df['voyage_id']=='8_255806337') | (df['voyage_id']=='9_255806337')]
dis = dis[['MMSI','PERIOD', 'time_diff', 'new_voyage', 'voyage_id']]

In [132]:
geo_df2.columns

Index(['MMSI', 'NAME', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS',
       'COG_DEG', 'HEADING_DEG', 'NAV_STATUS', 'SHIP_AND_CARGO_TYPE',
       'DRAUGHT', 'DIM_BOW', 'DIM_STERN', 'DIM_PORT', 'DIM_STARBOARD', 'BEAM',
       'LENGTH', 'CHANNEL_SIDE', 'time_diff', 'cog_diff', 'new_voyage',
       'voyage_id', 'include', 'geometry', 'in_channel_north',
       'in_channel_south', 'location', 'num_pings'],
      dtype='object')

In [39]:
voyage_list = list(geo_df2['voyage_id'].unique())

In [408]:
geo_df.to_csv('mid_datav1.csv')

In [40]:
print(geo_df2['MMSI'].nunique())
print(geo_df2['voyage_id'].nunique())
print(geo_df2.shape)

873
1897
(21148, 28)


In [67]:
my_map = folium.Map(location=[22.5, -77.8], zoom_start=9.45)
folium.GeoJson(channel_polygon_s, color='green').add_to(my_map)
folium.GeoJson(channel_polygon_n, color='orange').add_to(my_map)
folium.GeoJson(polygon_s, color='blue').add_to(my_map)
folium.GeoJson(polygon_n, color='red').add_to(my_map)
my_map

In [1]:
import random
random_voyages = random.sample(voyage_list, 20)
geo_df2['include'] = geo_df2['voyage_id'].isin(random_voyages)
random_voyages_gdf = geo_df2[geo_df2['include'] == True]
random_voyages_gdf.shape

NameError: name 'voyage_list' is not defined

In [47]:
def rand_color():
    return "#{:06x}".format(random.randint(0, 0xFFFFFF))

for id, vessel in random_voyages_gdf.groupby('voyage_id'):
    coords = list(vessel[['LAT_AVG', 'LON_AVG']].itertuples(index=False, name=None))
    if len(coords) < 2:
        continue

    folium.PolyLine(
        coords,
        color=rand_color(),
        weight=2,
        opacity=0.8,
        popup=f"MMSI: {id}"
    ).add_to(my_map)

my_map

In [352]:
geo_df2['MMSI'].nunique()

145

In [140]:
geo_df['num_pings'].describe()

count    65304.000000
mean        25.882274
std         31.295905
min          5.000000
25%          7.000000
50%         13.000000
75%         28.000000
max        167.000000
Name: num_pings, dtype: float64